# Lab Sheet 4: Fine-Tuning an LLM using LoRA/QLoRA

**Practical:** Fine-tune a small instruction-tuned LLM on a custom Q&A dataset using LoRA

**Tools:** Google Colab, Python, Hugging Face (`transformers`, `datasets`, `peft`, `trl`, `accelerate`, `bitsandbytes`)

**Model:** `Qwen/Qwen2.5-0.5B-Instruct`

**Domain:** Campus Essentials — a fictional university shop. This notebook follows Lab sheet 4 and
builds on the lecturer-provided example (`resources/fine_tune_generative_models.ipynb`), extended
to a full 30-question-and-answer dataset as the lab sheet requires (the provided example only had 10).

This notebook has not been run — run it top to bottom in Google Colab with a GPU runtime
(Runtime → Change runtime type → GPU).

In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


## Step 1 — Check the GPU

What GPU is available, and why does it matter for fine-tuning?

In [2]:
import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU Memory:", round(gpu.total_memory / 1024**3, 2), "GB")
else:
    print("No GPU available")

No GPU available


**Record — What GPU is available? Why is a GPU useful for LLM fine-tuning?**

The exact GPU depends on what Colab assigns at runtime (commonly a Tesla T4 with ~15 GB of memory
on the free tier). A GPU matters for fine-tuning because:

- Fine-tuning repeatedly runs the model forward (to get predictions) and backward (to compute
  gradients) over many training steps, and both passes are almost entirely large matrix
  multiplications — exactly the operation GPUs are built to do thousands of in parallel.
- A GPU's much higher memory bandwidth lets it move the model's weights and activations in and
  out of memory far faster than a CPU can.
- Without a GPU, the same fine-tuning run that takes minutes could take many hours, making
  iteration (trying different settings) impractical.

## Step 2 — Choose a small instruction-tuned model

`Qwen/Qwen2.5-0.5B-Instruct` — chosen because its ~0.5B parameters comfortably fit a free-tier Colab GPU, it is already instruction-tuned (so it follows a chat format out of the box), and it is one of the models suggested in the lab sheet.

In [3]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

## Step 3 — Build a 30-question-and-answer dataset about one domain

The lab sheet requires **at least 30** question/answer pairs about a specific domain. Continuing the lecturer's "Campus Essentials" university shop example, extended from 10 pairs to 30.

In [4]:
from datasets import Dataset

data = [
    {"question": "What products does Campus Essentials sell?",
     "answer": "Campus Essentials sells notebooks, pens, textbooks, USB drives, water bottles, and university merchandise."},
    {"question": "How much does a notebook cost?",
     "answer": "A standard notebook costs LKR 350."},
    {"question": "Do you sell computer science textbooks?",
     "answer": "Yes. Selected computer science textbooks are available, subject to stock availability."},
    {"question": "What time does the shop open?",
     "answer": "The shop opens at 8:30 AM from Monday to Friday."},
    {"question": "Can I return a damaged product?",
     "answer": "Yes. Customers can request an exchange for a damaged product by presenting the receipt within seven days of purchase."},
    {"question": "Do you sell USB drives?",
     "answer": "Yes. USB drives are available in selected storage capacities."},
    {"question": "Can I pay by card?",
     "answer": "Yes. The shop accepts cash and debit or credit cards."},
    {"question": "Are university-branded T-shirts available?",
     "answer": "Yes. University-branded T-shirts are available in selected sizes."},
    {"question": "Can I reserve a textbook?",
     "answer": "Yes. Customers can request a textbook reservation by contacting the shop. Reservations depend on stock availability."},
    {"question": "Where is Campus Essentials located?",
     "answer": "Campus Essentials is located beside the university library."},
    {"question": "Is the shop open on weekends?",
     "answer": "Yes. The shop is open on Saturday from 9:00 AM to 2:00 PM and is closed on Sunday."},
    {"question": "Do you sell calculators?",
     "answer": "Yes. Scientific calculators suitable for engineering and science students are available."},
    {"question": "How much does a scientific calculator cost?",
     "answer": "A scientific calculator costs LKR 2,500."},
    {"question": "Do you offer printing services?",
     "answer": "Yes. Both black-and-white and colour printing are available at the counter."},
    {"question": "How much is black-and-white printing per page?",
     "answer": "Black-and-white printing costs LKR 10 per page."},
    {"question": "Do you sell laptop bags?",
     "answer": "Yes. A small range of laptop bags and sleeves is available."},
    {"question": "Is there a student discount?",
     "answer": "Yes. Students receive a 10% discount on stationery with a valid student ID."},
    {"question": "Do you accept mobile payment apps?",
     "answer": "Yes. Selected mobile wallet apps are accepted at checkout."},
    {"question": "Can I order items in bulk for a club or society?",
     "answer": "Yes. Contact the shop in advance to arrange a bulk order for a club or society."},
    {"question": "Do you sell whiteboard markers?",
     "answer": "Yes. Both whiteboard markers and permanent markers are available."},
    {"question": "Is there a loyalty card?",
     "answer": "Yes. Customers collect a stamp on every purchase and receive a discount after ten stamps."},
    {"question": "What happens if I lose my receipt?",
     "answer": "An exchange may still be possible with other proof of purchase, such as a bank statement, at the shop's discretion."},
    {"question": "Do you sell second-hand textbooks?",
     "answer": "Yes. A small second-hand textbook section is available, subject to availability."},
    {"question": "Can I pre-order a textbook that's out of stock?",
     "answer": "Yes. Pre-orders are accepted, and you will be contacted when the textbook arrives."},
    {"question": "Do you sell headphones or earphones?",
     "answer": "Yes. A small range of budget headphones and earphones is available."},
    {"question": "Is gift wrapping available?",
     "answer": "Yes. Gift wrapping is available for a small additional charge."},
    {"question": "Do you recycle old notebooks or electronics?",
     "answer": "Yes. There is a collection point in the shop for recycling paper and small electronics."},
    {"question": "How can I contact Campus Essentials?",
     "answer": "You can visit the shop beside the university library or call the number posted at the counter."},
    {"question": "Are there seasonal discounts during exam season?",
     "answer": "Yes. Selected stationery items are discounted during exam weeks."},
    {"question": "Do you sell graph paper and lab notebooks?",
     "answer": "Yes. Graph paper pads and lab notebooks are both in stock."},
]

print("Number of examples:", len(data))
dataset = Dataset.from_list(data)

Number of examples: 30


## Step 4 — Load the base model and tokenizer

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if torch.cuda.is_available():
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
else:
    dtype = torch.float32

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=dtype,
    device_map="auto"
)

print("Model loaded successfully!")
print("Parameters:", f"{model.num_parameters():,}")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
C:\Users\Dell\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

## Step 5 — Format the dataset with the model's chat template

Each question/answer pair is wrapped in the model's expected conversation format so it fine-tunes on examples that look like real conversations.

In [ ]:
def format_example(example):
    messages = [
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_dataset = dataset.map(format_example, remove_columns=dataset.column_names)
print(train_dataset[0]["text"])

## Step 6 — Configure LoRA

Freeze the base model and only train small adapter matrices inside the attention and feed-forward projections.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 7 — Fine-tune with TRL's SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

training_args = SFTConfig(
    output_dir="./campus_essentials_lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    bf16=False,
    gradient_checkpointing=True,
    max_length=512,
    dataset_text_field="text",
    report_to="none",
    seed=42
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer
)

trainer.train()

## Step 8 — Save the LoRA adapter

Only the small trainable adapter is saved, not a full copy of the base model.

In [ ]:
output_dir = "./campus_essentials_lora/final_adapter"

trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print("LoRA adapter saved to:", output_dir)

## Step 9 — Compare the original model against the fine-tuned model

Test with questions that were **not** in the 30-example training dataset.

In [ ]:
def generate_answer(model_to_use, question):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model_to_use.device)

    model_to_use.eval()
    with torch.no_grad():
        outputs = model_to_use.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [ ]:
import gc
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Held-out questions: none of these appear in the 30 training examples above.
test_questions = [
    "Do you sell laptop chargers?",
    "Can I buy stationery and textbooks from the same shop?",
    "Is there parking available near the shop?",
    "Do you offer refunds, or only exchanges?",
]

if "model" in globals():
    del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=dtype, device_map="auto"
)
fine_tuned_model = PeftModel.from_pretrained(base_model, output_dir)

original_answers = [generate_answer(base_model, q) for q in test_questions]
fine_tuned_answers = [generate_answer(fine_tuned_model, q) for q in test_questions]

import pandas as pd

results = pd.DataFrame({
    "Question": test_questions,
    "Original Model": original_answers,
    "Fine-tuned Model": fine_tuned_answers
})

display(results)

## Step 10 (optional) — BLEU score comparison

A rough, automatic way to compare answer quality against short reference answers. With only 30 training examples, scores for both models will likely be low — this is a sanity check, not a reliable quality measure.

In [ ]:
!pip install -q sacrebleu
import sacrebleu

reference_answers = [
    "Laptop chargers are not currently stocked at Campus Essentials.",
    "Yes, stationery and textbooks are both sold at Campus Essentials.",
]

original_bleu = sacrebleu.corpus_bleu(original_answers[:2], [reference_answers])
finetuned_bleu = sacrebleu.corpus_bleu(fine_tuned_answers[:2], [reference_answers])

print(f"Original BLEU:   {original_bleu.score:.2f}")
print(f"Fine-tuned BLEU: {finetuned_bleu.score:.2f}")